In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
import nltk

nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [6]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Tejas B
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [17]:
import pandas as pd

file1 = pd.read_csv("C:\\Users\\Tejas B V\\Downloads\\fake_or_real_news.csv")
file2 = pd.read_csv("C:\\Users\\Tejas B V\\Downloads\\FakeNewsNet.csv")

print("First dataset:")
print(file1.shape)
print(file1.head())

print("\nSecond dataset:")
print(file2.shape)
print(file2.head())


First dataset:
(6335, 4)
   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  

Second dataset:
(23196, 5)
                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  S

In [18]:
print(file1.columns)
print(file2.columns)

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')
Index(['title', 'news_url', 'source_domain', 'tweet_num', 'real'], dtype='object')


In [15]:
print(file2['real'].value_counts(dropna=False))

real
1    17441
0     5755
Name: count, dtype: int64


In [19]:
print(file1['label'].value_counts(dropna=False))
print(file2['real'].value_counts(dropna=False))

label
REAL    3171
FAKE    3164
Name: count, dtype: int64
real
1    17441
0     5755
Name: count, dtype: int64


In [21]:
file1.shape

(6335, 4)

In [23]:
file2.shape

(23196, 5)

In [25]:
file1.shape, file2.shape

((6335, 4), (23196, 5))

In [27]:
file1.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [29]:
file2.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [31]:
file1.isnull().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [33]:
file2.isnull().sum()

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64

In [35]:
print("Duplicates in first dataset:", file1.duplicated().sum())

print("Duplicates in second dataset:", file2.duplicated().sum())

Duplicates in first dataset: 0
Duplicates in second dataset: 137


In [38]:
file1=file1.fillna('')
file2=file2.fillna('')

In [40]:
file2['news_url'] = file2['news_url'].fillna('')
file2['source_domain'] = file2['source_domain'].fillna('')

print(file2.isnull().sum())
print(file1.dtypes)
print()
print(file2.dtypes)


title            0
news_url         0
source_domain    0
tweet_num        0
real             0
dtype: int64
Unnamed: 0     int64
title         object
text          object
label         object
dtype: object

title            object
news_url         object
source_domain    object
tweet_num         int64
real              int64
dtype: object


In [44]:
print(file1.columns.tolist())

['title', 'text', 'label']


In [47]:

print(file1.shape)
print(file2.shape)

(6335, 3)
(23196, 5)


In [50]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_conetent = stemmed_content.split()
    stemmed_content = [PorterStemmer().stem(word) for word in stemmed_conetent if not word in set(stopwords.words('english'))]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [53]:
file1['content'] = file1['text'].apply(stemming)
file2['content'] = file2['title'].apply(stemming)


In [54]:
print(file1[['text', 'content']].head())
print(file2[['title', 'content']].head())

                                                text  \
0  Daniel Greenfield, a Shillman Journalism Fello...   
1  Google Pinterest Digg Linkedin Reddit Stumbleu...   
2  U.S. Secretary of State John F. Kerry said Mon...   
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...   
4  It's primary day in New York and front-runners...   

                                             content  
0  daniel greenfield shillman journal fellow free...  
1  googl pinterest digg linkedin reddit stumbleup...  
2  u secretari state john f kerri said monday sto...  
3  kayde king kaydeek novemb lesson tonight dem l...  
4  primari day new york front runner hillari clin...  
                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  Sophia Bush Sends Sweet Birthday Message to 'O...   
3  Colombian singer Maluma sparks rumours of inap...   
4  Gossip Girl 10 Years Later: How Upper East Sid... 

In [56]:
X1 = file1['content'].values
Y1 = file1['label'].values

print(X1.shape)
print(Y1.shape)


(6335,)
(6335,)


In [58]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X1)
x1 = vectorizer.transform(X1) 

In [61]:
X2 = file2['content'].values
Y2 = file2['real'].values

print("Headlines:", X2.shape)
print("Labels:", Y2.shape)


Headlines: (23196,)
Labels: (23196,)


In [63]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X2)
x2 = vectorizer.transform(X2) 

In [65]:
print(x1)

  (0, 106)	0.02761351227547292
  (0, 212)	0.023325671822305483
  (0, 271)	0.054392547007153
  (0, 328)	0.060358339691341766
  (0, 357)	0.013921754610837712
  (0, 451)	0.020578335403474236
  (0, 578)	0.07906058794712593
  (0, 617)	0.019850889079147057
  (0, 620)	0.01867769280554434
  (0, 622)	0.044611051373187424
  (0, 648)	0.015493583340569357
  (0, 680)	0.019224695171899866
  (0, 978)	0.0267316264615063
  (0, 999)	0.020738114075644642
  (0, 1022)	0.0371485992478585
  (0, 1033)	0.014808690907292291
  (0, 1085)	0.028815013247480384
  (0, 1089)	0.018102417756284926
  (0, 1197)	0.022505695388032227
  (0, 1206)	0.010675511636980443
  (0, 1549)	0.02507693986049084
  (0, 1665)	0.025967702221880915
  (0, 1713)	0.041128098634750046
  (0, 1744)	0.027618563562476325
  (0, 1783)	0.018872240981032625
  :	:
  (6334, 39650)	0.02398512016423436
  (6334, 39971)	0.04071827532138649
  (6334, 40086)	0.03890236771964674
  (6334, 40158)	0.04886237540126324
  (6334, 40783)	0.05859961268767996
  (6334, 41309

In [67]:
print (x2)

  (0, 53)	0.2721149519149911
  (0, 576)	0.32205761515065545
  (0, 1524)	0.4060028523437122
  (0, 3739)	0.37202135470266046
  (0, 5246)	0.27372476382132127
  (0, 5895)	0.4060028523437122
  (0, 8975)	0.3165911519884338
  (0, 9020)	0.24249280188561428
  (0, 9287)	0.26647854102470714
  (0, 11959)	0.2217815349642914
  (1, 643)	0.31690546637483147
  (1, 986)	0.34762560771735407
  (1, 1710)	0.3978429114527965
  (1, 1975)	0.43378159796649224
  (1, 6573)	0.3420111696667716
  (1, 8315)	0.4067948402953306
  (1, 9079)	0.387076508000102
  (2, 1069)	0.19438975915734968
  (2, 1361)	0.37698546923989756
  (2, 1528)	0.3313932058546551
  (2, 1529)	0.2679606830015344
  (2, 2132)	0.22767646060985314
  (2, 3657)	0.25142880383632443
  (2, 5100)	0.22534648956135603
  (2, 5104)	0.2418360754414745
  :	:
  (23193, 9046)	0.32490265953129527
  (23193, 10239)	0.39707689058614987
  (23194, 3306)	0.3321442681266319
  (23194, 3706)	0.2587855648260621
  (23194, 3907)	0.2822937698497612
  (23194, 5165)	0.276602517531464

In [87]:
from sklearn.model_selection import train_test_split

# Split the first dataset
X1_train, X1_test, Y1_train, Y1_test = train_test_split(
    file1['content'].values,
    file1['label'].values,
    test_size=0.2,
    random_state=2,
    stratify=file1['label']
)

# Split the second dataset
X2_train, X2_test, Y2_train, Y2_test = train_test_split(
    file2['content'].values,
    file2['real'].values,
    test_size=0.2,
    random_state=2,
    stratify=file2['real']
)

print("First dataset training samples:", len(X1_train))
print("First dataset testing samples:", len(X1_test))

print("Second dataset training samples:", len(X2_train))
print("Second dataset testing samples:", len(X2_test))

First dataset training samples: 5068
First dataset testing samples: 1267
Second dataset training samples: 18556
Second dataset testing samples: 4640


In [89]:
# Vectorizer for the first dataset
vectorizer1 = TfidfVectorizer(max_features=50000)

X1_train_tfidf = vectorizer1.fit_transform(X1_train)
X1_test_tfidf = vectorizer1.transform(X1_test)

# Vectorizer for the second dataset
vectorizer2 = TfidfVectorizer(max_features=20000)

X2_train_tfidf = vectorizer2.fit_transform(X2_train)
X2_test_tfidf = vectorizer2.transform(X2_test)

print("First dataset TF-IDF:", X1_train_tfidf.shape)
print("Second dataset TF-IDF:", X2_train_tfidf.shape)

First dataset TF-IDF: (5068, 39715)
Second dataset TF-IDF: (18556, 11473)


In [91]:
# Train the first model
model1 = LogisticRegression()
model1.fit(X1_train_tfidf, Y1_train)

print("First model trained successfully!")

# Train the second model
model2 = LogisticRegression()
model2.fit(X2_train_tfidf, Y2_train)

print("Second model trained successfully!")

First model trained successfully!
Second model trained successfully!


In [93]:
# Evaluate the first model
pred1 = model1.predict(X1_test_tfidf)

print("FIRST MODEL: FULL ARTICLE TEXT")
print("Accuracy:", accuracy_score(Y1_test, pred1))
print(classification_report(Y1_test, pred1))

# Evaluate the second model
pred2 = model2.predict(X2_test_tfidf)

print("\nSECOND MODEL: HEADLINES")
print("Accuracy:", accuracy_score(Y2_test, pred2))
print(classification_report(Y2_test, pred2))

FIRST MODEL: FULL ARTICLE TEXT
Accuracy: 0.9187056037884768
              precision    recall  f1-score   support

        FAKE       0.90      0.94      0.92       633
        REAL       0.94      0.90      0.92       634

    accuracy                           0.92      1267
   macro avg       0.92      0.92      0.92      1267
weighted avg       0.92      0.92      0.92      1267


SECOND MODEL: HEADLINES
Accuracy: 0.8355603448275862
              precision    recall  f1-score   support

           0       0.80      0.45      0.58      1151
           1       0.84      0.96      0.90      3489

    accuracy                           0.84      4640
   macro avg       0.82      0.71      0.74      4640
weighted avg       0.83      0.84      0.82      4640



In [ ]:


X1_train_prediction = model1.predict(X1_train_tfidf)



X1_test_prediction = model1.predict(X1_test_tfidf)



training_data_accuracy1 = accuracy_score(Y1_train, X1_train_prediction)
testing_data_accuracy1 = accuracy_score(Y1_test, X1_test_prediction)

print("MODEL 1: FULL NEWS ARTICLES")
print("Accuracy on training data:", training_data_accuracy1 * 100, "%")
print("Accuracy on testing data:", testing_data_accuracy1 * 100, "%")


MODEL 1: FULL NEWS ARTICLES
Accuracy on training data: 95.2446724546172 %
Accuracy on testing data: 91.87056037884767 %


In [97]:
# Predictions on training data

X2_train_prediction = model2.predict(X2_train_tfidf)

# Predictions on testing data

X2_test_prediction = model2.predict(X2_test_tfidf)

# Calculate accuracy

training_data_accuracy2 = accuracy_score(Y2_train, X2_train_prediction)
testing_data_accuracy2 = accuracy_score(Y2_test, X2_test_prediction)

print("MODEL 2: NEWS HEADLINES")
print("Accuracy on training data:", training_data_accuracy2 * 100, "%")
print("Accuracy on testing data:", testing_data_accuracy2 * 100, "%")


MODEL 2: NEWS HEADLINES
Accuracy on training data: 85.9775813752964 %
Accuracy on testing data: 83.55603448275862 %


In [100]:
X_new = X1_test_tfidf[0]

prediction = model1.predict(X_new)

print(prediction)

if prediction[0] == 'REAL':
    print('The news is Real')
else:
    print('The news is Fake')

['REAL']
The news is Real


In [103]:
# Select one article from the test dataset

X_new = X1_test_tfidf[0]

# Predict the label

prediction = model1.predict(X_new)

# Display the result

print("Predicted label:", prediction[0])

if prediction[0] == 'REAL':
     print("The news is Real")
else:
     print("The news is Fake")


Predicted label: REAL
The news is Real


In [2]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# Create these once, outside the function
stop_words = set(stopwords.words('english'))
porter_stemmer = PorterStemmer()

def stemming(content):
    content = re.sub('[^a-zA-Z]', ' ', content)
    content = content.lower()
    words = content.split()

    stemmed_words = [
        porter_stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return ' '.join(stemmed_words)

In [3]:
news = input("Enter a news article: ")

cleaned_news = stemming(news)

news_tfidf = vectorizer1.transform([cleaned_news])

prediction = model1.predict(news_tfidf)

print("Predicted label:", prediction[0])

NameError: name 'vectorizer1' is not defined